In [ ]:
##########################
## ÚLTIMA VERSÃO 12052026
##########################


!pip install pandas numpy matplotlib seaborn scipy statsmodels scikit-learn openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import plotly.express as px

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# UNIFICAÇÃO DOS DADOS - 12 MESES QUE FORAM CRIADOS PELO CÓDIGO "ATUALIZAÇÃO DE BASES".

In [ ]:
import os
import glob

# Craindo diretórios
d2 = "/content/drive/My Drive/PNAD/CAGED/PORTO"
d3 = "/content/drive/My Drive/PNAD/CAGED/PORTO_OCUPADOS"

# Função auxiliar para ler todos os arquivos de um padrão
def ler_arquivos(diretorio, padrao):
    arquivos = glob.glob(os.path.join(diretorio, padrao))
    lista_df = [pd.read_excel(arq) for arq in arquivos]
    return pd.concat(lista_df, ignore_index=True)

# Lendo e unificando os arquivos
dados_MEDIA_PORTO         = ler_arquivos(d2, "*_media.xlsx")
dados_ADMITIDOS_PORTO     = ler_arquivos(d2, "*_NUMADMITIDO.xlsx")
dados_MEDIA_PORTO_OCUPADOS     = ler_arquivos(d3, "*_media.xlsx")
dados_ADMITIDOS_PORTO_OCUPADOS = ler_arquivos(d3, "*_NUMADMITIDO.xlsx")

In [ ]:
# Leitura do arquivo do INPC atualizado - último disponível.

import requests
import re
import json
import csv
import os
from collections import defaultdict

def download_inpc_csv():
    url = "https://www.dadosdemercado.com.br/indices/inpc"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    print(f"Acessando {url}...")
    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Erro ao acessar a página: {response.status_code}")
        return

    html_content = response.text
    match = re.search(r'const data = (\[\[.*?\]\]);', html_content, re.DOTALL)

    if not match:
        print("Não foi possível encontrar os dados no HTML.")
        return

    data_str = match.group(1)
    try:
        raw_data = json.loads(data_str)
    except json.JSONDecodeError as e:
        print(f"Erro ao decodificar os dados: {e}")
        return

    # Reorganizar os dados para o formato de matriz (Ano x Mês)
    processed_data = defaultdict(lambda: {str(i).zfill(2): '' for i in range(1, 13)})
    years = set()

    for item in raw_data:
        date_str = item[0]
        value = item[1]

        year = date_str[:4]
        month = date_str[5:7]

        processed_data[year][month] = str(value).replace('.', ',') + '%'
        years.add(year)

    sorted_years = sorted(list(years), reverse=True)
    months_order = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]
    month_names = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]

    # Salvar como CSV no caminho especificado
    save_directory = "/content/drive/My Drive/PNAD/CAGED"
    os.makedirs(save_directory, exist_ok=True)

    filename = os.path.join(save_directory, "indices.csv")
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)

        # Escrever cabeçalho
        header = ["Ano"] + month_names + ["Ano"]
        writer.writerow(header)

        # Escrever dados
        for year in sorted_years:
            row = [year]
            for month in months_order:
                row.append(processed_data[year].get(month, '--')) # Usar '--' para meses sem dados

            # Calcular a soma anual se houver dados para o ano
            annual_sum = 0.0
            has_data_for_year = False
            for month_val in processed_data[year].values():
                if month_val and month_val != '--':
                    try:
                        # Remover '%' e substituir ',' por '.' para conversão
                        clean_val = month_val.replace('%', '').replace(',', '.')
                        annual_sum += float(clean_val)
                        has_data_for_year = True
                    except ValueError:
                        pass # Ignorar valores não numéricos

            if has_data_for_year:
                row.append(f"{annual_sum:.2f}%".replace('.', ','))
            else:
                row.append('--')

            writer.writerow(row)

    print(f"Arquivo '{filename}' salvo com sucesso!")
    print(f"Total de anos processados: {len(sorted_years)}")

if __name__ == "__main__":
    download_inpc_csv()

Acessando https://www.dadosdemercado.com.br/indices/inpc...
Arquivo '/content/drive/My Drive/PNAD/CAGED/indices.csv' salvo com sucesso!
Total de anos processados: 27


In [ ]:
#### ORGANIZANDO O ARQUIVO DE INPC PARA SE JUNTAS AS BASE DE MÉDIAS (SÃO 3 BASES)

INPC = pd.read_csv("/content/drive/My Drive/PNAD/CAGED/indices.csv")

# Transformar de wide para long (Jan-Dez viram uma coluna 'mes')
inpc = INPC.melt(
    id_vars=["Ano"],
    value_vars=["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"],
    var_name="mes",
    value_name="inpc")

# Renomear categorias de mês para números
mapa_meses = {
    "Jan":"01","Fev":"02","Mar":"03","Abr":"04","Mai":"05","Jun":"06",
    "Jul":"07","Ago":"08","Set":"09","Out":"10","Nov":"11","Dez":"12"}
inpc["mes"] = inpc["mes"].map(mapa_meses)

# 5. Concatenar ano+mes
inpc["Anomes"] = inpc["Ano"].astype(str) + inpc["mes"]

# 6. Selecionar apenas colunas desejadas
inpc = inpc[["Anomes", "inpc"]]

In [ ]:
########################################################
#### Preparando os bancos de MÉDIAS para agregar o INPC
########################################################

# Renomeando colunas
dados_MEDIA_PORTO = dados_MEDIA_PORTO.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "X": "Media_Salario" })

dados_MEDIA_PORTO_OCUPADOS = dados_MEDIA_PORTO_OCUPADOS.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "X": "Media_Salario" })

# Ajustando categorias da coluna 'contrato'
mapa_contrato = {-1: "Demitidos", 1: "Admitidos"}

dados_MEDIA_PORTO["contrato"] = dados_MEDIA_PORTO["contrato"].map(mapa_contrato)
dados_MEDIA_PORTO_OCUPADOS["contrato"] = dados_MEDIA_PORTO_OCUPADOS["contrato"].map(mapa_contrato)

In [ ]:
###############################################################
#### Organizando os bancos de Número de admitidos para plotagem
###############################################################

# Renomeando colunas
dados_ADMITIDOS_PORTO = dados_ADMITIDOS_PORTO.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "n": "n"  })

dados_ADMITIDOS_PORTO_OCUPADOS = dados_ADMITIDOS_PORTO_OCUPADOS.rename(
    columns={
        "competênciamov": "Anomes",
        "saldomovimentação": "contrato",
        "n": "n" })

# Ajustando categorias da coluna 'contrato'
mapa_contrato = {-1: "Demitidos", 1: "Admitidos"}

dados_ADMITIDOS_PORTO["contrato"] = dados_ADMITIDOS_PORTO["contrato"].map(mapa_contrato)
dados_ADMITIDOS_PORTO_OCUPADOS["contrato"] = dados_ADMITIDOS_PORTO_OCUPADOS["contrato"].map(mapa_contrato)

In [ ]:
### AGREGANDO INPC NAS BASES DE MÉDIAS por ano/mes
################################################################################

# Garantir que 'Anomes' seja string em todas as bases
dados_MEDIA_PORTO["Anomes"] = dados_MEDIA_PORTO["Anomes"].astype(str)
dados_MEDIA_PORTO_OCUPADOS["Anomes"] = dados_MEDIA_PORTO_OCUPADOS["Anomes"].astype(str)

inpc["Anomes"] = inpc["Anomes"].astype(str)

# Merge com INPC
dados_MEDIA_PORTO = pd.merge(dados_MEDIA_PORTO, inpc, on="Anomes")
dados_MEDIA_PORTO_OCUPADOS = pd.merge(dados_MEDIA_PORTO_OCUPADOS, inpc, on="Anomes")

# Função auxiliar para limpar INPC
def limpar_inpc(df):
    df["inpc"] = (
        df["inpc"]
        .astype(str)                # garante que é string
        .str.replace(",", ".", regex=False)  # troca vírgula por ponto
        .str.replace("%", "", regex=False)   # remove símbolo de porcentagem
    )
    df["inpc"] = pd.to_numeric(df["inpc"], errors="coerce")  # converte para numérico
    return df

# 3. Aplicar limpeza em cada base
dados_MEDIA_PORTO = limpar_inpc(dados_MEDIA_PORTO)
dados_MEDIA_PORTO_OCUPADOS = limpar_inpc(dados_MEDIA_PORTO_OCUPADOS)

In [ ]:
#### AGREGANDO INPC ACUMULADO e DEFLACIONAMENTO
####  Porto Feliz E ocupados
#################################################

def calcular_inpc(df):
    # 1. Converter INPC em fator (1 + inpc/100)
    df["inpc_fator"] = 1 + df["inpc"] / 100

    # 2. Calcular INPC acumulado (produto cumulativo)
    df["inpc_acm"] = df["inpc_fator"].cumprod()

    # 3. Calcular salário deflacionado
    df["Sal_def_INPC"] = df["Media_Salario"] / df["inpc_acm"]

    return df

# Aplicar para cada base
dados_MEDIA_PORTO = calcular_inpc(dados_MEDIA_PORTO)
dados_MEDIA_PORTO_OCUPADOS = calcular_inpc(dados_MEDIA_PORTO_OCUPADOS)

In [ ]:
###  GRÁFICO 3 - SALÁRIO MÉDIO DEFLACIONADO PELO INPC - PORTO FELIZ

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_MEDIA_PORTO["Anomes"],
    "categoria": dados_MEDIA_PORTO["contrato"],
    "inpc": dados_MEDIA_PORTO["Sal_def_INPC"]
})

# Converter de long para wide (pivot)
SP3_inpc = temporario.pivot(index="time", columns="categoria", values="inpc")

# Converter coluna de tempo para formato de data
SP3_inpc.index = pd.to_datetime(SP3_inpc.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(SP3_inpc, x=SP3_inpc.index, y=SP3_inpc.columns,
              title="Salário médio deflacionado pelo INPC - Porto Feliz (sem filtro de ocupação)")

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/Porto_média.html", include_plotlyjs="cdn")

fig.show()

In [ ]:
###  GRÁFICO 4 - NÚMERO DE ADMITIDOS/DEMITIDOS - PORTO FELIZ

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_ADMITIDOS_PORTO["Anomes"],
    "categoria": dados_ADMITIDOS_PORTO["contrato"],
    "n": dados_ADMITIDOS_PORTO["n"]
})

# Converter de long para wide
SP4_n = temporario.pivot(index="time", columns="categoria", values="n")

# Converter coluna de tempo para formato de data
SP4_n.index = pd.to_datetime(SP4_n.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(SP4_n, x=SP4_n.index, y=SP4_n.columns,
              title="Número de Admitidos e Demitidos - Porto Feliz (sem filtro de ocupação)")

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/Porto_Admitidos.html", include_plotlyjs="cdn")

fig.show()

In [ ]:
###  GRÁFICO 5 - SALÁRIO MÉDIO DEFLACIONADO PELO INPC - PORTO FELIZ OCUPADOS

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_MEDIA_PORTO_OCUPADOS["Anomes"],
    "categoria": dados_MEDIA_PORTO_OCUPADOS["contrato"],
    "inpc": dados_MEDIA_PORTO_OCUPADOS["Sal_def_INPC"]
})

# Converter de long para wide (pivot)
SP5_inpc = temporario.pivot(index="time", columns="categoria", values="inpc")

# Converter coluna de tempo para formato de data
SP5_inpc.index = pd.to_datetime(SP5_inpc.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(SP5_inpc, x=SP5_inpc.index, y=SP5_inpc.columns,
              title="Salário médio deflacionado pelo INPC - Porto Feliz (com filtro de ocupação)")

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/Porto_Ocupados_média.html", include_plotlyjs="cdn")

fig.show()

In [ ]:
###  GRÁFICO 6 - NÚMERO DE ADMITIDOS/DEMITIDOS - PORTO FELIZ OCUPADOS

# Criar dataframe temporário
temporario = pd.DataFrame({
    "time": dados_ADMITIDOS_PORTO_OCUPADOS["Anomes"],
    "categoria": dados_ADMITIDOS_PORTO_OCUPADOS["contrato"],
    "n": dados_ADMITIDOS_PORTO_OCUPADOS["n"]
})

# Converter de long para wide
SP6_n = temporario.pivot(index="time", columns="categoria", values="n")

# Converter coluna de tempo para formato de data
SP6_n.index = pd.to_datetime(SP6_n.index.astype(str) + "01", format="%Y%m%d")

# Gráfico interativo
fig = px.line(SP6_n, x=SP6_n.index, y=SP6_n.columns,
              title="Número de Admitidos e Demitidos - Porto Feliz (com filtro de ocupação)")

# Salvar gráfico como HTML no Drive
fig.write_html("/content/drive/My Drive/PNAD/GRAFICOS/Porto_Ocupados_Admitidos.html", include_plotlyjs="cdn")

fig.show()